In [ ]:
import glob, os, shutil, subprocess, tempfile
from pathlib import Path
OCTAVE = shutil.which("octave-cli") or shutil.which("octave")
if OCTAVE is None and os.name == "nt":
    candidates = glob.glob(r"C:\Program Files\GNU Octave\Octave-*\mingw64\bin\octave-cli.exe")
    if candidates:
        OCTAVE = sorted(candidates)[-1]
if OCTAVE is None:
    raise RuntimeError("Octave CLI not found. Add octave-cli to PATH or install GNU Octave.")
WORK = Path(tempfile.mkdtemp(prefix="me256-coursework-"))
print(f"Octave: {OCTAVE}")
print(f"Temporary source directory: {WORK}")

for relative in ['cylindrical-mirror', 'plane-mirror', 'plane-refraction', 'reflection-geometry', 'spherical-mirror']:
    (WORK / relative).mkdir(parents=True, exist_ok=True)


# ME256 Homework 2: reflection and refraction

**Official statement:** [ME256 Homework 2 (2011)](https://mecheng.iisc.ac.in/suresh/me256/hw2011/homework2.html). The outline below paraphrases the tasks for study; use the linked page for the complete statement and exact requirements.

The assignment asks you to formulate and solve four geometric-optics models, then use sufficient-condition analysis to determine whether each computed stationary point is a local minimum, local maximum, or saddle point:

1. A ray reflecting from a straight line between two points in a plane.
2. Reflection from a concave spherical mirror between two points in space.
3. Reflection from a right circular cylinder between two points in space.
4. Refraction between two media, with the endpoints in a plane and on opposite sides of the interface.

The archived MATLAB programs mainly explore path-length/travel-time objectives, numerical candidate points, and plots. They do **not** by themselves provide all the stationary-point classification and sufficient-condition reasoning requested by the assignment; treat those as study questions to complete alongside the code.

## Study prompts

- Which coordinates parameterize the point where a ray meets each surface, and what feasible set do those coordinates have?
- What objective represents optical path length or travel time in each model?
- How do you use first- and second-order conditions to classify a stationary point, including possible competing extrema?

## Reproducibility

This notebook uses a Python 3 Jupyter kernel to materialize the embedded MATLAB source into a temporary directory, then invokes Octave CLI. The `.m` sources are embedded below; they are not stored separately in this repository. Install Jupyter/IPython and an Octave installation with the `optim` package. MATLAB with Optimization Toolbox remains the closest match to the original runtime.

### Review note

The maximum-path helper `cylindrical-mirror/reflection_cylinder1.m` contains a suspicious endpoint-coordinate term. It is intentionally preserved and flagged in the assignment README; compare it with the original code PDF before drawing conclusions. A corrected experiment, if desired, should be a separate variant.

## Embedded source code

Each following code cell contains one MATLAB source file and writes it into a temporary working directory. Run the notebook from top to bottom; no reconstructed `.m` files are required in the repository.

### `reflection-geometry/geometry1.m`

In [ ]:
%%writefile {WORK / 'reflection-geometry' / 'geometry1.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Sweep points on a circular mirror and compare path length and ray angles.
clc;clear all;close all;
mirror_angle=[0:pi/100:49*pi/100 51*pi/100:pi/100:149*pi/100 151*pi/100:pi/100:2*pi];
source_point=[.3;-.5];                             %Ray emerges from this point
destination_point=[.5;.5];                              %Ray reflects to this point
mirror_radius=1;
reflection_points=[mirror_radius*cos(mirror_angle);mirror_radius*sin(mirror_angle)];          %Point on periphery
total_path_length=sqrt(source_point(1)^2+source_point(2)^2+mirror_radius^2-2*mirror_radius*(source_point(1)*cos(mirror_angle)+source_point(2)*sin(mirror_angle)))+sqrt(destination_point(1)^2+destination_point(2) ^2+mirror_radius^2-2*mirror_radius*(destination_point(1)*cos(mirror_angle)+destination_point(2)*sin(mirror_angle)));
%Distance traversed by ray
plot(mirror_angle*180/pi,total_path_length*50)                 %Law of Reflection holds when d is stationary w.r.to theta
%d upscaled to prominence in graph
hold on
source_ray_slope=(source_point(2)-mirror_radius*sin(mirror_angle))./(source_point(1)-mirror_radius*cos(mirror_angle));
destination_ray_slope=(destination_point(2)-mirror_radius*sin(mirror_angle))./(destination_point(1)-mirror_radius*cos(mirror_angle));
mirror_tangent_slope=tan(mirror_angle);
incidence_angle=(atan((source_ray_slope-mirror_tangent_slope)./(1+source_ray_slope.*mirror_tangent_slope)));            %Angle of Incidence
reflection_angle=(atan((destination_ray_slope-mirror_tangent_slope)./(1+destination_ray_slope.*mirror_tangent_slope)));            %Angle of Reflection
plot(mirror_angle*180/pi,incidence_angle*180/pi,'r')
hold on
plot(mirror_angle*180/pi,reflection_angle*180/pi,'g')
hold on
plot(mirror_angle*180/pi,(incidence_angle+reflection_angle)*180/pi,'c')     %Law of Reflection holdss when (i+r)=0
grid on
xlabel('theta');
legend('distance(x50)','Angle of Incidence','Angle of Reflection','i+r');


### `plane-mirror/Reflection_PlaneMirror_2D.m`

In [ ]:
%%writefile {WORK / 'plane-mirror' / 'Reflection_PlaneMirror_2D.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Find a reflection point on a straight mirror by minimizing total path length.
clc;clear all;close all;
global source_point destination_point
mirror_slope=2;
mirror_intercept=3;
mirror_equation=@(mirror_x,mirror_y) mirror_y-mirror_slope*mirror_x-mirror_intercept;                        %Equation of Mirror
source_point=[1;2];                                 %Ray emerges from this point
destination_point=[3;7];                                 %Ray reflected to this point
initial_reflection_point=[0;mirror_intercept];
if(source_point(2)-mirror_slope*source_point(1)-mirror_intercept)*(destination_point(2)-mirror_slope*destination_point(1)-mirror_intercept)<0
    display('Points are opposite sides of mirror; Reflection is not possible.')
else
    [reflection_point,path_length,exit_flag]=fmincon(@reflection,initial_reflection_point,[],[],[-mirror_slope 1],mirror_intercept,[],[],[],optimset('Algorithm','active-set'))
    plot_x_min=min(source_point(1),destination_point(1))-2;
    plot_x_max=max(source_point(1),destination_point(1))+2;
    plot_y_min=min(source_point(2),destination_point(2))-2;
    plot_y_max=max(source_point(2),destination_point(2))+2;
    ezplot(mirror_equation,[plot_x_min,plot_x_max,plot_y_min,plot_y_max])
    hold on
    line([source_point(1),reflection_point(1),destination_point(1)],[source_point(2),reflection_point(2),destination_point(2)],'Color','r')
    plot(source_point(1),source_point(2),'Marker','o')
    text(source_point(1)+.15,source_point(2),'x');
    plot(destination_point(1),destination_point(2),'Marker','o')
    text(destination_point(1)+.15,destination_point(2),'y');
    plot(reflection_point(1),reflection_point(2),'Marker','o')
    text(reflection_point(1)+.15,reflection_point(2),'z');
    legend('Mirror','Reflection Path x->z->y')
    title('Reflection on Plane Mirror')
end
incidence_angle=(atan(-1/mirror_slope)-atan((source_point(2)-reflection_point(2))/(source_point(1)-reflection_point(1))))*180/pi;
reflection_angle=(atan((destination_point(2)-reflection_point(2))/(destination_point(1)-reflection_point(1)))-atan(-1/mirror_slope))*180/pi;
if incidence_angle<0
    incidence_angle=incidence_angle+180;
end
if reflection_angle<0
    reflection_angle=reflection_angle+180;
end
incidence_angle
reflection_angle


### `plane-mirror/reflection.m`

In [ ]:
%%writefile {WORK / 'plane-mirror' / 'reflection.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Objective: sum the distances from the two ray endpoints to the mirror point.
function total_path_length=reflection(reflection_point)
    global source_point destination_point
    total_path_length=norm(source_point-reflection_point,2)+norm(destination_point-reflection_point,2);


### `spherical-mirror/Reflection_SphericalMirror.m`

In [ ]:
%%writefile {WORK / 'spherical-mirror' / 'Reflection_SphericalMirror.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Compare minimum- and maximum-path reflection points on a circular mirror.
clc;clear all;close all;
global source_point destination_point mirror_radius
mirror_radius=1;
mirror_equation=@(mirror_x,mirror_y) mirror_x^2+mirror_y^2-mirror_radius^2;                %Equation of Spherical Mirror
%Origin set at the centre
source_point=[-.5;.7];                          %Ray emerges from this point
destination_point=[.5;.5];                           %Ray reflected to this point
initial_mirror_angle=1;
if source_point(1)^2+source_point(2)^2>mirror_radius^2
    display('Point x is outside of the mirror')
else
    if destination_point(1)^2+destination_point(2)^2>mirror_radius^2
        display('Point y is outside of the mirror')
    else
        [minimum_path_angle,minimum_path_length,optimizer_exit_flag]=fmincon(@reflection_spherical,initial_mirror_angle,[],[],[], [],-pi,pi,[],optimset('Algorithm','active-set'))
        %Reflection corresponding to min{d}
        [maximum_path_angle,negative_maximum_path_length,optimizer_exit_flag]=fmincon(@reflection_spherical1,initial_mirror_angle,[], [],[],[],-pi,pi,[],optimset('Algorithm','active-set'))
        %Reflection corresponding to max{d}
        minimum_reflection_point=[mirror_radius*cos(minimum_path_angle);mirror_radius*sin(minimum_path_angle)];
        maximum_reflection_point=[mirror_radius*cos(maximum_path_angle);mirror_radius*sin(maximum_path_angle)];
        ezplot(mirror_equation)
        hold on
        line([source_point(1),minimum_reflection_point(1),destination_point(1)],[source_point(2),minimum_reflection_point(2),destination_point(2)],'Color','r')
        hold on
        line([source_point(1),maximum_reflection_point(1),destination_point(1)],[source_point(2),maximum_reflection_point(2),destination_point(2)],'Color','c')
        plot(source_point(1),source_point(2),'Marker','o')
        text(source_point(1)+mirror_radius/20,source_point(2),'x');
        plot(destination_point(1),destination_point(2),'Marker','o')
        text(destination_point(1)+mirror_radius/20,destination_point(2),'y');
        plot(minimum_reflection_point(1),minimum_reflection_point(2),'Marker','o')
        text(minimum_reflection_point(1)+mirror_radius/20,minimum_reflection_point(2),'z_dmin');
        plot(maximum_reflection_point(1),maximum_reflection_point(2),'Marker','o')
        text(maximum_reflection_point(1)+mirror_radius/20,maximum_reflection_point(2),'z_dmax');
        axis square
        axis tight
        xlabel('x')
        ylabel('y')
        legend('Mirror','Reflection x>dmin>y','Reflection x>dmax>y')
        title('Reflection on Concave Spherical Mirror')
        incidence_angle_minimum_path=angle((source_point(1)-minimum_reflection_point(1)+(source_point(2)-minimum_reflection_point(2))*i)/(-minimum_reflection_point(1) -minimum_reflection_point(2)*i))*180/pi
        reflection_angle_minimum_path=angle((-minimum_reflection_point(1)-minimum_reflection_point(2)*i)/(destination_point(1)-minimum_reflection_point(1)+(destination_point(2) -minimum_reflection_point(2))*i))*180/pi
        incidence_angle_maximum_path=angle((source_point(1)-maximum_reflection_point(1)+(source_point(2)-maximum_reflection_point(2))*i)/(-maximum_reflection_point(1) -maximum_reflection_point(2)*i))*180/pi
        reflection_angle_maximum_path=angle((-maximum_reflection_point(1)-maximum_reflection_point(2)*i)/(destination_point(1)-maximum_reflection_point(1)+(destination_point(2) -maximum_reflection_point(2))*i))*180/pi
    end
end


### `spherical-mirror/reflection_spherical.m`

In [ ]:
%%writefile {WORK / 'spherical-mirror' / 'reflection_spherical.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Total path length for a reflection point parameterized by its angle.
function path_length=reflection_spherical(mirror_angle)
    global source_point destination_point mirror_radius
    path_length=sqrt(source_point(1)^2+source_point(2)^2+mirror_radius^2-2*mirror_radius*(source_point(1)*cos(mirror_angle)+source_point(2)*sin(mirror_angle)))+sqrt(destination_point(1)^2+destination_point(2) ^2+mirror_radius^2-2*mirror_radius*(destination_point(1)*cos(mirror_angle)+destination_point(2)*sin(mirror_angle)));


### `spherical-mirror/reflection_spherical1.m`

In [ ]:
%%writefile {WORK / 'spherical-mirror' / 'reflection_spherical1.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Negated path length lets fmincon seek a maximum path length.
function negative_path_length=reflection_spherical1(mirror_angle)
    global source_point destination_point mirror_radius
    negative_path_length=-sqrt(source_point(1)^2+source_point(2)^2+mirror_radius^2-2*mirror_radius*(source_point(1)*cos(mirror_angle)+source_point(2)*sin(mirror_angle)))-sqrt(destination_point(1)^2+destination_point(2) ^2+mirror_radius^2-2*mirror_radius*(destination_point(1)*cos(mirror_angle)+destination_point(2)*sin(mirror_angle)));


### `cylindrical-mirror/Reflection_Cylindrical.m`

In [ ]:
%%writefile {WORK / 'cylindrical-mirror' / 'Reflection_Cylindrical.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Optimize reflection points on a finite cylindrical mirror.
clc;clear all;close all;
global source_point destination_point cylinder_radius
cylinder_radius=1;
source_point=[-.5;.1;.1];                          %Ray emerges from this point
destination_point=[.5;-.5;.3];                           %Ray reflected to this point
initial_reflection_parameters=[1;.2];
if source_point(1)^2+source_point(2)^2>cylinder_radius^2
    display('Point x is outside of the mirror')
else
    if destination_point(1)^2+destination_point(2)^2>cylinder_radius^2
        display('Point y is outside of the mirror')
    else
        [minimum_reflection_parameters,minimum_path_length,optimizer_exit_flag]=fmincon(@reflection_cylinder,initial_reflection_parameters,[],[],[],[],[0;min(source_point(3),destination_point(3))],[2*pi;max(source_point(3),destination_point(3))],[],optimset('Algorithm','active-set'))
        %Reflection corresponding to min{d}
        [maximum_reflection_parameters,maximum_search_objective,optimizer_exit_flag]=fmincon(@reflection_cylinder1,initial_reflection_parameters,[],[],[],[],[- pi;min(source_point(3),destination_point(3))],[pi;max(source_point(3),destination_point(3))],[],optimset('Algorithm','active-set'))
        %Reflection corresponding to max{d}
        [surface_x,surface_y,surface_z] = cylinder(cylinder_radius,200);
        surface_z(1,:)=min(source_point(3),destination_point(3));
        surface_z(2,:)=max(source_point(3),destination_point(3));
        surf(surface_x,surface_y,surface_z)
        colormap winter
        shading interp
        if ~exist('OCTAVE_VERSION', 'builtin') || exist('alpha', 'file')
            alpha(.3)
        end
        hold on
        line([source_point(1),cylinder_radius*cos(minimum_reflection_parameters(1)),destination_point(1)],[source_point(2),cylinder_radius*sin(minimum_reflection_parameters(1)),destination_point(2)],[source_point(3), minimum_reflection_parameters(2),destination_point(3)],'Color','r')
        hold on
        line([source_point(1),cylinder_radius*cos(maximum_reflection_parameters(1)),destination_point(1)],[source_point(2),cylinder_radius*sin(maximum_reflection_parameters(1)),destination_point(2)],[source_point(3), maximum_reflection_parameters(2),destination_point(3)],'Color','y')
        plot3(source_point(1),source_point(2),source_point(3),'Marker','o')
        text(source_point(1)+cylinder_radius/20,source_point(2),source_point(3),'x');
        plot3(destination_point(1),destination_point(2),destination_point(3),'Marker','o')
        text(destination_point(1)+cylinder_radius/20,destination_point(2),destination_point(3),'y');
        plot3(cylinder_radius*cos(minimum_reflection_parameters(1)),cylinder_radius*sin(minimum_reflection_parameters(1)),minimum_reflection_parameters(2),'Marker','o')
        text(cylinder_radius*cos(minimum_reflection_parameters(1))+cylinder_radius/20,cylinder_radius*sin(minimum_reflection_parameters(1)),minimum_reflection_parameters(2),'z_dmin');
        plot3(cylinder_radius*cos(maximum_reflection_parameters(1)),cylinder_radius*sin(maximum_reflection_parameters(1)),maximum_reflection_parameters(2),'Marker','o')
        text(cylinder_radius*cos(maximum_reflection_parameters(1))+cylinder_radius/20,cylinder_radius*sin(maximum_reflection_parameters(1)),maximum_reflection_parameters(2),'z_dmax');
        axis square
        axis tight
        xlabel('x')
        ylabel('y')
        zlabel('z')
        legend('Mirror','Reflection x>z_dmin>y','Reflection x>z_dmax>y')
        title('Reflection on Cylindrical Mirror')
        incidence_angle_minimum_path=subspace([(source_point(1)-cylinder_radius*cos(minimum_reflection_parameters(1)));(source_point(2)-cylinder_radius*sin(minimum_reflection_parameters(1)));(source_point(3)-minimum_reflection_parameters(2))],[cylinder_radius*cos(minimum_reflection_parameters(1));cylinder_radius*sin(minimum_reflection_parameters(1));0])*180/pi
        reflection_angle_minimum_path=subspace([cylinder_radius*cos(minimum_reflection_parameters(1));cylinder_radius*sin(minimum_reflection_parameters(1));0],[(destination_point(1)-cylinder_radius*cos(minimum_reflection_parameters(1)));(destination_point(2)-cylinder_radius*sin(minimum_reflection_parameters(1)));(destination_point(3)-minimum_reflection_parameters(2))])*180/pi
        incidence_angle_maximum_path=subspace([(source_point(1)-cylinder_radius*cos(maximum_reflection_parameters(1)));(source_point(2)-cylinder_radius*sin(maximum_reflection_parameters(1)));(source_point(3)-maximum_reflection_parameters(2))],[cylinder_radius*cos(maximum_reflection_parameters(1));cylinder_radius*sin(maximum_reflection_parameters(1));0])*180/pi
        reflection_angle_maximum_path=subspace([cylinder_radius*cos(maximum_reflection_parameters(1));cylinder_radius*sin(maximum_reflection_parameters(1));0],[(destination_point(1)-cylinder_radius*cos(maximum_reflection_parameters(1)));(destination_point(2)-cylinder_radius*sin(maximum_reflection_parameters(1)));(destination_point(3)-maximum_reflection_parameters(2))])*180/pi
    end
end


### `cylindrical-mirror/Reflection_Cylindrical_Extern.m`

In [ ]:
%%writefile {WORK / 'cylindrical-mirror' / 'Reflection_Cylindrical_Extern.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Repeat the cylindrical-mirror search with endpoints outside the cylinder.
clc;clear all;close all;
global source_point destination_point cylinder_radius
cylinder_radius=1;
source_point=[5;7;.1];                          %Ray emerges from this point
destination_point=[0;-5;.3];                           %Ray reflected to this point
initial_reflection_parameters=[1;.2];
if source_point(1)^2+source_point(2)^2<cylinder_radius^2
    display('Point x is inside of the mirror')
else
    if destination_point(1)^2+destination_point(2)^2<cylinder_radius^2
        display('Point y is inside of the mirror')
    else
        [minimum_reflection_parameters,minimum_path_length,optimizer_exit_flag]=fmincon(@reflection_cylinder,initial_reflection_parameters,[],[],[],[],[-pi; min(source_point(3),destination_point(3))],[pi;max(source_point(3),destination_point(3))],[],optimset('Algorithm','active-set'))
        %Reflection corresponding to min{d}
        [maximum_reflection_parameters,maximum_search_objective,optimizer_exit_flag]=fmincon(@reflection_cylinder1,initial_reflection_parameters,[],[],[],[],[- pi;min(source_point(3),destination_point(3))],[pi;max(source_point(3),destination_point(3))],[],optimset('Algorithm','active-set'))
        %Reflection corresponding to max{d}
        [surface_x,surface_y,surface_z] = cylinder(cylinder_radius,200);
        surface_z(1,:)=min(source_point(3),destination_point(3));
        surface_z(2,:)=max(source_point(3),destination_point(3));
        surf(surface_x,surface_y,surface_z)
        colormap copper
        shading interp
        if ~exist('OCTAVE_VERSION', 'builtin') || exist('alpha', 'file')
            alpha(.5)
        end
        hold on
        line([source_point(1),cylinder_radius*cos(minimum_reflection_parameters(1)),destination_point(1)],[source_point(2),cylinder_radius*sin(minimum_reflection_parameters(1)),destination_point(2)],[source_point(3), minimum_reflection_parameters(2),destination_point(3)],'Color','r')
        hold on
        line([source_point(1),cylinder_radius*cos(maximum_reflection_parameters(1)),destination_point(1)],[source_point(2),cylinder_radius*sin(maximum_reflection_parameters(1)),destination_point(2)],[source_point(3), maximum_reflection_parameters(2),destination_point(3)],'Color','g')
        plot3(source_point(1),source_point(2),source_point(3),'Marker','o')
        text(source_point(1)+cylinder_radius/20,source_point(2),source_point(3),'x');
        plot3(destination_point(1),destination_point(2),destination_point(3),'Marker','o')
        text(destination_point(1)+cylinder_radius/20,destination_point(2),destination_point(3),'y');
        plot3(cylinder_radius*cos(minimum_reflection_parameters(1)),cylinder_radius*sin(minimum_reflection_parameters(1)),minimum_reflection_parameters(2),'Marker','o')
        text(cylinder_radius*cos(minimum_reflection_parameters(1))+cylinder_radius/20,cylinder_radius*sin(minimum_reflection_parameters(1)),minimum_reflection_parameters(2),'z_dmin');
        plot3(cylinder_radius*cos(maximum_reflection_parameters(1)),cylinder_radius*sin(maximum_reflection_parameters(1)),maximum_reflection_parameters(2),'Marker','o')
        text(cylinder_radius*cos(maximum_reflection_parameters(1))+cylinder_radius/20,cylinder_radius*sin(maximum_reflection_parameters(1)),maximum_reflection_parameters(2),'z_dmax');
        set(gca,'DataAspectRatio',[1 1 .02])
        xlabel('x')
        ylabel('y')
        zlabel('z')
        legend('Mirror','Reflection x>z_dmin>y','Reflection x>z_dmax>y')
        title('Reflection on Cylindrical Mirror')
        incidence_angle_minimum_path=subspace([(source_point(1)-cylinder_radius*cos(minimum_reflection_parameters(1)));(source_point(2)-cylinder_radius*sin(minimum_reflection_parameters(1)));(source_point(3)-minimum_reflection_parameters(2))],[cylinder_radius*cos(minimum_reflection_parameters(1));cylinder_radius*sin(minimum_reflection_parameters(1));0])*180/pi
        reflection_angle_minimum_path=subspace([cylinder_radius*cos(minimum_reflection_parameters(1));cylinder_radius*sin(minimum_reflection_parameters(1));0],[(destination_point(1)-cylinder_radius*cos(minimum_reflection_parameters(1)));(destination_point(2)-cylinder_radius*sin(minimum_reflection_parameters(1)));(destination_point(3)-minimum_reflection_parameters(2))])*180/pi
        incidence_angle_maximum_path=subspace([(source_point(1)-cylinder_radius*cos(maximum_reflection_parameters(1)));(source_point(2)-cylinder_radius*sin(maximum_reflection_parameters(1)));(source_point(3)-maximum_reflection_parameters(2))],[cylinder_radius*cos(maximum_reflection_parameters(1));cylinder_radius*sin(maximum_reflection_parameters(1));0])*180/pi
        reflection_angle_maximum_path=subspace([cylinder_radius*cos(maximum_reflection_parameters(1));cylinder_radius*sin(maximum_reflection_parameters(1));0],[(destination_point(1)-cylinder_radius*cos(maximum_reflection_parameters(1)));(destination_point(2)-cylinder_radius*sin(maximum_reflection_parameters(1)));(destination_point(3)-maximum_reflection_parameters(2))])*180/pi
    end
end


### `cylindrical-mirror/reflection_cylinder.m`

In [ ]:
%%writefile {WORK / 'cylindrical-mirror' / 'reflection_cylinder.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Sum the two endpoint-to-mirror distances for a cylindrical reflection.
function path_length=reflection_cylinder(reflection_parameters)
    global source_point destination_point cylinder_radius
    path_length=sqrt(source_point(1)^2+source_point(2)^2+cylinder_radius^2-2*cylinder_radius*(source_point(1)*cos(reflection_parameters(1))+source_point(2)*sin(reflection_parameters(1)))+(source_point(3)-reflection_parameters(2))^2)+sqrt(destination_point(1)^2+destination_point(2)^2+cylinder_radius^2-2*cylinder_radius*(destination_point(1)*cos(reflection_parameters(1))+destination_point(2)*sin(reflection_parameters(1)))+(destination_point(3)-reflection_parameters(2))^2);


### `cylindrical-mirror/reflection_cylinder1.m`

In [ ]:
%%writefile {WORK / 'cylindrical-mirror' / 'reflection_cylinder1.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Reciprocal-distance objective used for the maximum-path search.
function reciprocal_path_objective=reflection_cylinder1(reflection_parameters)
    global source_point destination_point cylinder_radius
    % Preserve the source expression: its second endpoint term uses source_point(3).
    path_length=sqrt(source_point(1)^2+source_point(2)^2+cylinder_radius^2-2*cylinder_radius*(source_point(1)*cos(reflection_parameters(1))+source_point(2)*sin(reflection_parameters(1)))+(source_point(3)-reflection_parameters(2))^2)+sqrt(destination_point(1)^2+destination_point(2)^2+cylinder_radius^2-2*cylinder_radius*(destination_point(1)*cos(reflection_parameters(1))+destination_point(2)*sin(reflection_parameters(1)))+(source_point(3)-reflection_parameters(2))^2);
    reciprocal_path_objective=1/path_length;


### `plane-refraction/refraction_plane.m`

In [ ]:
%%writefile {WORK / 'plane-refraction' / 'refraction_plane.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Minimize travel time across a planar interface between two media.
clc;clear all;close all;
global source_point destination_point source_medium_speed destination_medium_speed
positive_side_speed=1;                                  %Speed of Light at +ve side of plane of refraction
negative_side_speed=2;                                 %Speed of Light at -ve side of plane of refraction
interface_slope=0;
interface_intercept=0;
interface_equation=@(interface_x,interface_y) interface_y-interface_slope*interface_x-interface_intercept;                        %Equation of Plane of Refraction
source_point=[6;6];                                 %Ray emerges from this point
destination_point=[-5;-5];                                %Ray refracted to this point
endpoint_line_slope=(source_point(2)-destination_point(2))/(source_point(1)-destination_point(1));
endpoint_line_intercept=(source_point(1)*destination_point(2)-source_point(2)*destination_point(1))/(source_point(1)-destination_point(1));
initial_refraction_point=[-interface_slope,1;-endpoint_line_slope,1]\[interface_intercept;endpoint_line_intercept];
source_side_distance=source_point(2)-interface_slope*source_point(1)-interface_intercept;
destination_side_distance=destination_point(2)-interface_slope*destination_point(1)-interface_intercept;
if source_side_distance*destination_side_distance>0
    display('Points are same sides of mirror; Refraction is not possible.')
else
    if source_side_distance>0
        source_medium_speed=positive_side_speed;destination_medium_speed=negative_side_speed;
    else
        source_medium_speed=negative_side_speed;destination_medium_speed=positive_side_speed;
    end
    [refraction_point,travel_time,exit_flag]=fmincon(@refraction,initial_refraction_point,[],[],[-interface_slope 1],interface_intercept,[],[],[],optimset('Algorithm','active-set'))
    plot_x_min=min(source_point(1),destination_point(1))-2;
    plot_x_max=max(source_point(1),destination_point(1))+2;
    plot_y_min=min(source_point(2),destination_point(2))-2;
    plot_y_max=max(source_point(2),destination_point(2))+2;
    ezplot(interface_equation,[plot_x_min,plot_x_max,plot_y_min,plot_y_max])
    hold on
    line([source_point(1),refraction_point(1),destination_point(1)],[source_point(2),refraction_point(2),destination_point(2)],'Color','r')
    plot(source_point(1),source_point(2),'Marker','o')
    text(source_point(1)+.15,source_point(2),'x');
    plot(destination_point(1),destination_point(2),'Marker','o')
    text(destination_point(1)+.15,destination_point(2),'y');
    plot(refraction_point(1),refraction_point(2),'Marker','o')
    text(refraction_point(1)+.15,refraction_point(2),'z');
    legend('Plane of Refraction','Refraction Path x->z->y')
    title('Refraction on Plane Mirror')
end
incidence_angle=(atan(-1/interface_slope)-atan((source_point(2)-refraction_point(2))/(source_point(1)-refraction_point(1))))*180/pi;
refraction_angle=(atan(-1/interface_slope)-atan((destination_point(2)-refraction_point(2))/(destination_point(1)-refraction_point(1))))*180/pi;
if incidence_angle<0
    incidence_angle=incidence_angle+180;
end
if refraction_angle<0
    refraction_angle=refraction_angle+180;
end
incidence_angle
refraction_angle
sind(incidence_angle)/sind(refraction_angle)
source_medium_speed/destination_medium_speed


### `plane-refraction/refraction.m`

In [ ]:
%%writefile {WORK / 'plane-refraction' / 'refraction.m'}
% Reconstructed from the assignment2 source-code PDF.
% Problem-solving approach is retained; see the assignment README.
% Travel-time objective using the speeds on each side of the interface.
function travel_time=refraction(refraction_point)
    global source_point destination_point source_medium_speed destination_medium_speed
    travel_time=(norm(source_point-refraction_point,2))/source_medium_speed+(norm(destination_point-refraction_point,2))/destination_medium_speed;


## Octave 11.3.0 test results

All six entry-point scripts completed with exit code 0 using Octave `optim` 1.6.3. The captured cell output contains numerical stationary points and solver exit flags. Octave may use different solver behavior and plotting APIs from MATLAB; the suspicious cylindrical maximum-objective formula remains unchanged and is documented in the assignment README.

In [ ]:
import subprocess

def run_case(label, folder, script, needs_optim=True):
    setup = "set(0,'defaultfigurevisible','off');"
    if needs_optim:
        setup += "pkg load optim;"
    setup += "if any(strcmp(available_graphics_toolkits(),'gnuplot')), graphics_toolkit('gnuplot'); end;"
    setup += f"run('{script}');"
    result = subprocess.run(
        [OCTAVE, "--no-gui", "--quiet", "--eval", setup],
        cwd=WORK / folder, text=True, capture_output=True, errors="replace"
    )
    status = "PASS" if result.returncode == 0 else "FAILED"
    print(f"\n=== {label}: {status} (exit code {result.returncode}) ===")
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
    return result.returncode

run_case('Circular reflection geometry', 'reflection-geometry', 'geometry1.m', False)
run_case('Plane mirror', 'plane-mirror', 'Reflection_PlaneMirror_2D.m', True)
run_case('Spherical mirror', 'spherical-mirror', 'Reflection_SphericalMirror.m', True)
run_case('Cylindrical mirror', 'cylindrical-mirror', 'Reflection_Cylindrical.m', True)
run_case('External cylindrical reflection', 'cylindrical-mirror', 'Reflection_Cylindrical_Extern.m', True)
run_case('Plane refraction', 'plane-refraction', 'refraction_plane.m', True)



=== Circular reflection geometry: PASS (exit code 0) ===
--- stderr ---

The gnuplot graphics toolkit is not actively maintained and has a number
of limitations that are unlikely to be fixed.  Communication with gnuplot
uses a one-directional pipe and limited information is passed back to the
Octave interpreter so most changes made interactively in the plot window
will not be reflected in the graphics properties managed by Octave.  For
example, if the plot window is closed with a mouse click, Octave will not
be notified and will not update its internal list of open figure windows.
The qt toolkit is recommended instead.

The FLTK graphics toolkit is not actively maintained and has a number
of limitations that are unlikely to be fixed.
The qt toolkit is recommended instead.

=== Plane mirror: PASS (exit code 0) ===
reflection_point =

   1.2400
   5.4800

path_length = 5.8138
exit_flag = 1
incidence_angle = 67.380
reflection_angle = 67.380
--- stderr ---

The gnuplot graphics toolkit is

In [ ]:
import shutil
shutil.rmtree(WORK)
print("Temporary generated source files removed.")